In [4]:
import pandas as pd

# 1. Find AD, PD and commont proteins
AD_prots = set(pd.read_csv('../networks/AD_nodes.csv')['name'].to_list())
PD_prots = set(pd.read_csv('../networks/PD_nodes.csv')['name'].to_list())
common_prots = AD_prots.intersection(PD_prots)

# 2. Add label colummn
sim_matrix = pd.read_csv('../output/similarity_matrix_all_mean_cleaned.csv')
sim_matrix = sim_matrix.rename(columns={"Unnamed: 0": "name"})
sim_matrix['features'] = sim_matrix.iloc[:, 1:].values.tolist()
sim_matrix = sim_matrix[['name', 'features']]
sim_matrix['label'] = sim_matrix.iloc[:, 0].apply(lambda x: 2 if x in common_prots else (0 if x in AD_prots else 1))

sim_matrix.to_csv('../pre_processing_output/classification_three_labels_nodes_sim.csv', index_label='id')

# 3. Exclude minor components's proteins (given the PPI of AD and PD)
import pickle as pkl
with open('../pre_processing_output/minor_components_prots.pkl', 'rb') as serialized:
    minor_component_prots = set(pkl.load(serialized))

sim_matrix_minor_comps = sim_matrix[~sim_matrix['name'].isin(minor_component_prots)]
sim_matrix_minor_comps = sim_matrix_minor_comps.rename(columns={'name': 'STRING_id'})
sim_matrix_minor_comps.to_csv('../pre_processing_output/classification_three_labels_nodes_sim_main_components.csv', index_label='id')

In [5]:
sim_matrix_minor_comps

,STRING_id,features,label
2,9606.ENSP00000377296,"[0.315, 0.343666666666667, 1.0, 0.422666666666...",2
3,9606.ENSP00000428968,"[0.288333333333333, 0.305333333333333, 0.42266...",1
4,9606.ENSP00000317473,"[0.259, 0.192666666666667, 0.308666666666667, ...",1
5,9606.ENSP00000447300,"[0.395666666666667, 0.362, 0.354333333333333, ...",1
6,9606.ENSP00000366070,"[0.355, 0.289666666666667, 0.277, 0.3603333333...",1
...,...,...,...
170,9606.ENSP00000377015,"[0.524666666666667, 0.539666666666667, 0.356, ...",0
171,9606.ENSP00000451300,"[0.35, 0.402666666666667, 0.263, 0.283, 0.221,...",0
172,9606.ENSP00000351926,"[0.443333333333333, 0.495666666666667, 0.31366...",0
177,9606.ENSP00000272317,"[0.321333333333333, 0.312, 0.290666666666667, ...",0


In [6]:
# Here I duplicate common proteins to create two disjoint graphs, one for AD and one for PD
# Minor components are considered outliers, thus excluded.

nodes_df = pd.read_csv('../pre_processing_output/duplicated_nodes_main_components.csv')

sim_matrix_dup = sim_matrix.copy()
sim_matrix_dup[sim_matrix_dup['label'] == 2]

df_ad = sim_matrix_dup[sim_matrix_dup['label'] == 2].copy()
df_ad['name'] += '_AD'
df_ad['label'] = 0

df_pd = sim_matrix_dup[sim_matrix_dup['label'] == 2].copy()
df_pd['name'] += '_PD'
df_pd['label'] = 1

sim_matrix_dup = sim_matrix_dup[sim_matrix_dup['label'] != 2]
sim_matrix_dup = pd.concat([sim_matrix_dup, df_ad, df_pd], ignore_index=True)

name_to_sim = dict(zip(sim_matrix_dup['name'], sim_matrix_dup['features']))
nodes_df['GO_embeddings'] = nodes_df['STRING_id'].map(name_to_sim)
nodes_df.to_csv('../pre_processing_output/duplicated_nodes_main_components_sim.csv')